In [1]:
import os
import shutil
import random
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from PIL import Image
import torch.nn.functional as F
from sklearn.metrics import classification_report


In [2]:
# === 2. Setup Transfer Learning with the split folders ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
base_dir = 'Splited_Data'                   

dataset = datasets.ImageFolder('aug_processed_data', transform=transform)
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model_resnet = models.resnet18(pretrained=True)
for param in model_resnet.parameters():
    param.requires_grad = False  # Freeze backbone

num_features = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_features, len(train_dataset.classes))  # Number of classes

model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_resnet.fc.parameters(), lr=0.001)

c:\Users\HP\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\HP\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:12<00:00, 3.75MB/s]


In [3]:
# === Training loop ===
def train_model(epochs=10):
    best_val_acc = 0.0
    for epoch in range(epochs):
        model_resnet.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model_resnet(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_loss = total_loss / len(train_loader)

        model_resnet.eval()
        val_correct, val_total, val_loss = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model_resnet(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    print(f"\n🏆 Best Val Accuracy: {best_val_acc:.2f}%")

In [4]:
# === Run training ===
train_model(epochs=10)

Epoch 1 | Train Loss: 0.7094 | Train Acc: 53.12% | Val Loss: 0.7962 | Val Acc: 57.50%
Epoch 2 | Train Loss: 0.5757 | Train Acc: 66.88% | Val Loss: 0.5287 | Val Acc: 80.00%
Epoch 3 | Train Loss: 0.4571 | Train Acc: 78.12% | Val Loss: 0.3872 | Val Acc: 75.00%
Epoch 4 | Train Loss: 0.4244 | Train Acc: 83.75% | Val Loss: 0.4009 | Val Acc: 87.50%
Epoch 5 | Train Loss: 0.3699 | Train Acc: 88.12% | Val Loss: 0.3312 | Val Acc: 90.00%
Epoch 6 | Train Loss: 0.3011 | Train Acc: 91.88% | Val Loss: 0.2888 | Val Acc: 92.50%
Epoch 7 | Train Loss: 0.2751 | Train Acc: 90.62% | Val Loss: 0.2612 | Val Acc: 95.00%
Epoch 8 | Train Loss: 0.2435 | Train Acc: 95.00% | Val Loss: 0.2574 | Val Acc: 92.50%
Epoch 9 | Train Loss: 0.2637 | Train Acc: 92.50% | Val Loss: 0.2496 | Val Acc: 92.50%
Epoch 10 | Train Loss: 0.2508 | Train Acc: 91.88% | Val Loss: 0.2166 | Val Acc: 92.50%

🏆 Best Val Accuracy: 95.00%


Model Testing

In [5]:
def predict_single_image(image_path, model, class_names):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        _, predicted = torch.max(probs, 1)

    print(f"Predicted Class: {class_names[predicted.item()]}")
    print(f"Class Probabilities: {probs.squeeze().numpy()}")

In [6]:
# Assuming dataset = ImageFolder(...)
class_names = dataset.classes  # ['healthy', 'infected']

# Path to one test image
test_image_path_1 = "processed_data/serie infected leaves/infected_05.png"

predict_single_image(test_image_path_1, model_resnet, class_names)

Predicted Class: series_infected_leaves
Class Probabilities: [0.29871058 0.7012894 ]


In [7]:
test_image_path_2 = "processed_data/serie healthy leaves/healthy_05.png"
predict_single_image(test_image_path_2, model_resnet, class_names)

Predicted Class: serie_healthy_leaves
Class Probabilities: [0.9755996  0.02440038]


Model Evaluation

In [9]:
def evaluate_final_model():
    model_resnet.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_resnet(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Final Evaluation on Validation Set:")
    print(classification_report(all_labels, all_preds, target_names=val_dataset.classes, digits=2))

# Run this after training
print("Evaluation of Resnet Model")
evaluate_final_model()

Evaluation of Resnet Model

📊 Final Evaluation on Validation Set:
                        precision    recall  f1-score   support

  serie_healthy_leaves       0.95      0.90      0.92        20
series_infected_leaves       0.90      0.95      0.93        20

              accuracy                           0.93        40
             macro avg       0.93      0.93      0.92        40
          weighted avg       0.93      0.93      0.92        40

